# [6.2] Gemma Scope Deep Dive - Solutions

This notebook runs the solved Gemma Scope validation contracts and then displays the report-backed CUDA signature result. Keep the claim boundary in view: this is a pinned artifact plus one real-activation validation split, not a broad claim that Gemma Scope features are all understood.

<details>
<summary>Expected output</summary>

The local tests should all print pass messages, and the final table/plots should match the committed `verification_report.json` metrics.

</details>

<details>
<summary>Help - why these controls matter</summary>

Released feature artifacts are easy to overread. Metadata, score reductions, AUC controls, deltas, ablation controls, steering guards, and DLA each remove one common way to fool yourself.

</details>


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part2_gemma_scope_deep_dive"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_deep_dive.tests as tests
import part2_gemma_scope_deep_dive.solutions as solutions

FeatureArtifactMetadata = solutions.FeatureArtifactMetadata
TaggedFeatureSpec = solutions.TaggedFeatureSpec
metadata_is_complete = solutions.metadata_is_complete
features_with_tag = solutions.features_with_tag
feature_score_vector = solutions.feature_score_vector
roc_auc_binary = solutions.roc_auc_binary
validate_feature_scores = solutions.validate_feature_scores
base_instruction_feature_delta = solutions.base_instruction_feature_delta
ablation_control_report = solutions.ablation_control_report
steering_safety_report = solutions.steering_safety_report
direct_logit_attribution = solutions.direct_logit_attribution
run_smoke_test = solutions.run_smoke_test


## Metadata And Score Tests

These tests cover artifact metadata, tag filtering, and prompt-level feature-score reductions.


In [ ]:
tests.test_metadata_completeness_and_tag_selection(
    FeatureArtifactMetadata,
    TaggedFeatureSpec,
    metadata_is_complete,
    features_with_tag,
)
tests.test_feature_score_vector_reductions_match_reference(feature_score_vector)


## Validation And Causal-Control Tests

These tests cover AUC validation, base-vs-instruction deltas, ablation controls, steering guards, direct logit attribution, and the whole local smoke-test contract.


In [ ]:
tests.test_validate_feature_scores_beats_baseline_and_reports_means(
    validate_feature_scores,
    roc_auc_binary,
)
tests.test_base_instruction_delta_reports_signed_and_abs_change(
    base_instruction_feature_delta,
)
tests.test_ablation_control_requires_target_ablation_to_beat_random(
    ablation_control_report,
)
tests.test_steering_safety_report_checks_control_and_perplexity_guard(
    steering_safety_report,
)
tests.test_direct_logit_attribution_matches_selected_token_projection(
    direct_logit_attribution,
)
tests.test_notebook_contract(run_smoke_test)


## Signature Result

<details>
<summary>Interpreting the signature result</summary>

The pinned Gemma Scope SAE loads, the CUDA forward path runs, and authenticated Gemma 3 layer-13 activations produce a held-out feature AUC of `1.000` against a random-feature AUC of `0.500` and label-shuffle AUC of `0.000`. This is a narrow benign split validation, not broad semantic or causal proof.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    real = gpu["gemma_scope_real_activation_preflight"]
    return [
        ("Gemma Scope repo", gpu["gemma_scope_repo_id"]),
        ("artifact", gpu["gemma_scope_artifact_path"]),
        ("revision", gpu["gemma_scope_revision"][:12]),
        ("shape", f"d_model {gpu['gemma_scope_d_model']}, width {gpu['gemma_scope_width']}"),
        ("encoder / decoder", f"{gpu['gemma_scope_w_enc_shape']} / {gpu['gemma_scope_w_dec_shape']}"),
        ("CUDA forward", gpu["gemma_scope_forward_passed"]),
        ("base model authenticated", gpu["gemma3_base_authenticated"]),
        ("residual / feature-score shape", f"{real['residual_shape']} / {real['feature_score_shape']}"),
        ("train / held-out prompts", f"{real['train_prompt_count']} / {real['heldout_prompt_count']}"),
        ("selected / random feature", f"{real['selected_feature_id']} / {real['random_control_feature_id']}"),
        ("held-out AUC", round(real["feature_auc"], 3)),
        ("random-feature AUC", round(real["baseline_auc"], 3)),
        ("label-shuffle AUC", round(real["label_shuffle_auc"], 3)),
        ("threshold accuracy", round(real["threshold_accuracy"], 3)),
        ("positive / negative mean", f"{real['positive_mean']:.2f} / {real['negative_mean']:.2f}"),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
real = gpu["gemma_scope_real_activation_preflight"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["feature", "random", "shuffle"],
    [real["feature_auc"], real["baseline_auc"], real["label_shuffle_auc"]],
    color=["#2563eb", "#94a3b8", "#f97316"],
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Held-out AUC controls")
axes[0].set_ylabel("AUC")

axes[1].bar(
    ["technical", "narrative"],
    [real["positive_mean"], real["negative_mean"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title(f"Feature {real['selected_feature_id']} scores")
axes[1].set_ylabel("mean activation")

axes[2].bar(
    ["artifact", "real activations"],
    [gpu["gemma_scope_peak_vram_gb"], real["peak_vram_gb"]],
    color=["#7c3aed", "#0f766e"],
)
axes[2].set_title("Peak CUDA memory")
axes[2].set_ylabel("GB")

fig.tight_layout()
plt.show()


## Limitations

The report proves a scoped Gemma Scope artifact and real-activation validation path. It does not prove broad Gemma Scope feature semantics, safety-relevant steering, refusal behavior, or real-model causal ablation. The strongest real-model result is one small benign technical-vs-narrative split with random-feature and label-shuffle controls.
